# Module 6 lab · Neural networks: how they compute

The code from every lesson of [this module](https://aicourses.dev/research/neural-networks), in order. Read a lesson, then run its cells here: each lesson builds on the one before, so run the cells from the top the first time.

**First time?** Read the [one-time setup](https://aicourses.dev/research/labs): a Google account, your own copy, and what to expect.

The lessons that explain this code are part of [AI Courses Pro](https://aicourses.dev/#pricing); the module’s first lesson is free.

Under each cell, *The lesson shows* is what it should print. Your last digits can differ a little.

**Keep your work:** File → Save a copy in Drive. Colab does not keep changes to this copy.

## 6.2 One neuron: weights, a bias and a sum

A neuron multiplies each input by a weight, adds them up, adds one more number called the bias, and passes the result on. This bias has nothing to do with the data bias of Module 0. It is Module 5’s dot product with a job to do.

[Read the lesson](https://aicourses.dev/research/neural-networks/one-neuron)

### Three lines of Python

In [ ]:
# how hungry, how tired, how much money is left (each 0 to 1)
inputs  = [0.9, 0.6, 0.3]
weights = [2.0, 1.5, 1.0]   # how much each one matters
bias = -2.5                 # your usual reluctance to order

z = sum(w * x for w, x in zip(weights, inputs)) + bias
print(z)                                             # 0.5
print("order pizza" if z > 0 else "cook at home")    # order pizza

NumPy does the multiply-and-add for you in one call, which is also much faster when there are thousands of inputs:

In [ ]:
import numpy as np

z = np.dot(weights, inputs) + bias
print(float(z))   # 0.5 (float makes it a plain number)

PyTorch, which you met in Module 5, has a ready-made neuron. `torch.nn.Linear(3, 1)` holds three weights and one bias:

In [ ]:
import torch

# 3 inputs and 1 output: 3 weights and 1 bias
neuron = torch.nn.Linear(3, 1)
print(neuron.weight.shape, neuron.bias.shape)
# torch.Size([1, 3]) torch.Size([1])

## 6.3 Activation functions: the bend that makes networks powerful

Without a bend, any stack of layers collapses into one straight line. ReLU, sigmoid and tanh add the bend, and you watch a straight line fail where a bent one succeeds.

[Read the lesson](https://aicourses.dev/research/neural-networks/activation-functions)

### The bends in code

In [ ]:
import numpy as np

def relu(z):
    return np.maximum(0, z)      # keep positives, negatives become 0

def sigmoid(z):
    return 1 / (1 + np.exp(-z))  # squash any number into 0 to 1

z = np.array([-2.0, 0.0, 0.5, 2.0])
print(relu(z))                   # [0.  0.  0.5 2. ]
print(sigmoid(z).round(3))       # [0.119 0.5   0.622 0.881]
print(np.tanh(z).round(3))       # [-0.964  0.     0.462  0.964]

`np.maximum(0, z)` compares every number with 0 and keeps the bigger one, the whole array at once, and `np.tanh` is NumPy’s ready-made tanh. The shower network is two lines more:

In [ ]:
temps = np.array([25, 30, 34, 38, 42, 46])   # shower temperature, °C
h1 = relu(temps - 30)        # wakes up above 30 °C
h2 = relu(temps - 38)        # wakes up above 38 °C
comfort = 0.25 * h1 - 0.5 * h2
print(comfort)               # [0. 0. 1. 2. 1. 0.]

PyTorch has the same functions ready-made:

In [ ]:
import torch

t = torch.tensor([-2.0, 0.0, 0.5, 2.0])
print(torch.relu(t))      # tensor([0.0000, 0.0000, 0.5000, 2.0000])
print(torch.sigmoid(t))   # tensor([0.1192, 0.5000, 0.6225, 0.8808])

## 6.4 Layers: many neurons at once, as one matrix multiply

A layer is many neurons reading the same inputs, which is exactly a matrix times a vector. It is why GPUs, built for matrix multiplication, run neural networks so well.

[Read the lesson](https://aicourses.dev/research/neural-networks/layers-as-matrices)

### A layer in code

In [ ]:
import numpy as np

x = np.array([0.9, 0.6, 0.3])          # hunger, tiredness, money
W = np.array([[ 2.0,  1.5,  1.0],      # pizza neuron’s weights
              [-1.0, -2.0, -1.0]])     # cook neuron’s weights
b = np.array([-2.5, 2.0])              # one bias per neuron

z = W @ x + b
print(z)                          # [ 0.5 -0.4]
print(W.shape, x.shape, z.shape)  # (2, 3) (3,) (2,)

`@` is matrix multiplication. For the batch, put the nights in rows and multiply by `W.T`, NumPy’s transpose:

In [ ]:
X = np.array([[0.9, 0.6, 0.3],    # tonight
              [0.2, 0.1, 0.8],    # a lazy Sunday
              [0.5, 0.9, 0.5],    # after a long shift
              [0.7, 0.3, 0.1]])   # payday is far away
Z = X @ W.T + b
print(Z.shape)    # (4, 2)
print(Z[1])       # [-1.15  0.8 ]

PyTorch’s `nn.Linear` is exactly this layer. It stores $W$ in the same shape, one row per neuron, and computes $XW^{\mathsf{T}} + \mathbf{b}$:

In [ ]:
import torch

torch.manual_seed(0)
layer = torch.nn.Linear(3, 2)        # 3 inputs, 2 neurons
print(layer.weight.shape, layer.bias.shape)
# torch.Size([2, 3]) torch.Size([2])
nights = torch.rand(4, 3)            # 4 nights, 3 numbers each
print(layer(nights).shape)           # torch.Size([4, 2])

## 6.5 The forward pass: from input to answer, layer by layer

Stacking layers turns pixels or words into an answer. Tracing the shapes through each layer is how you will read any architecture diagram later, the transformer included.

[Read the lesson](https://aicourses.dev/research/neural-networks/forward-pass)

### The forward pass in code

First in NumPy, so nothing is hidden. `rng.normal(0, 0.05, shape)` fills an array with random numbers centred on 0 with a spread of about 0.05; they stand in for weights training has not set yet.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

X = rng.random((32, 28, 28))     # 32 fake pictures, 28 × 28 pixels
X = X.reshape(32, 784)           # flatten each picture into one row

W1 = rng.normal(0, 0.05, (128, 784))   # layer 1: 128 neurons
b1 = np.zeros(128)
W2 = rng.normal(0, 0.05, (10, 128))    # layer 2: 10 neurons
b2 = np.zeros(10)

H = np.maximum(0, X @ W1.T + b1)       # layer 1, then ReLU
S = H @ W2.T + b2                      # ten scores per picture
print(X.shape, H.shape, S.shape)       # (32, 784) (32, 128) (32, 10)

PyTorch’s `nn.Sequential` chains layers so that each one’s output feeds the next. Looping over it lets you print the shape after every step:

In [ ]:
import torch
from torch import nn

torch.manual_seed(0)
model = nn.Sequential(
    nn.Flatten(),          # each 28 × 28 picture becomes 784 numbers
    nn.Linear(784, 128),   # 128 neurons
    nn.ReLU(),
    nn.Linear(128, 10),    # 10 scores, one per kind of clothing
)
x = torch.rand(32, 28, 28)
for layer in model:
    x = layer(x)
    print(type(layer).__name__, tuple(x.shape))
print(sum(p.numel() for p in model.parameters()))

## 6.6 From scores to probabilities: softmax, and what the output means

A network outputs raw scores. Softmax turns them into probabilities you can read, sample from and compare with the right answer.

[Read the lesson](https://aicourses.dev/research/neural-networks/scores-to-probabilities)

### Softmax in code

In [ ]:
import numpy as np

def softmax(scores):
    e = np.exp(scores)           # every score becomes positive
    return e / e.sum()           # divide by the total so they add to 1

scores = np.array([2.0, 1.0, 0.1])     # pizza, curry, salad
p = softmax(scores)
print(p.round(3), p.sum())             # [0.659 0.242 0.099] 1.0
print(softmax(scores + 10).round(3))   # [0.659 0.242 0.099]
print(softmax(scores / 2).round(3))    # [0.502 0.304 0.194]

rng = np.random.default_rng(0)
picks = rng.choice(3, size=1000, p=p)  # 1,000 random dinners
print(np.bincount(picks))              # [637 268  95]

For a batch, PyTorch’s `torch.softmax` needs to know which direction holds one example’s scores. With one night per row, that is `dim=1`, across each row:

In [ ]:
import torch

S = torch.tensor([[2.0, 1.0, 0.1],     # night 1: pizza, curry, salad
                  [0.5, 0.5, 3.0]])    # night 2
print(torch.softmax(S, dim=1))
# tensor([[0.6590, 0.2424, 0.0986],
#         [0.0705, 0.0705, 0.8590]])
print(torch.softmax(S, dim=0))         # wrong: mixes the two nights
# tensor([[0.8176, 0.6225, 0.0522],
#         [0.1824, 0.3775, 0.9478]])

## 6.7 Loss: putting a number on how wrong the network is

Learning needs one number to make smaller. Squared error for numbers and cross-entropy for choices are the two losses this course uses, and you compute both by hand.

[Read the lesson](https://aicourses.dev/research/neural-networks/loss-functions)

### Both losses in code

In [ ]:
import numpy as np

guess  = np.array([30.0, 25.0, 40.0])   # predicted delivery, minutes
actual = np.array([35.0, 24.0, 50.0])   # how long it really took
errors = guess - actual
print(errors)                    # [ -5.   1. -10.]
print((errors ** 2).mean())      # 42.0

p = np.array([0.659, 0.242, 0.099])     # pizza, curry, salad
print(-np.log(p).round(3))       # [0.417 1.419 2.313]

`np.log` is the natural log. `torch.nn.functional`, imported as `F`, holds PyTorch’s ready-made functions, such as losses. Its `F.cross_entropy` takes the **raw scores**, not probabilities, and does the softmax itself, so you hand it the network’s output directly, with the index of the right answer:

In [ ]:
import torch
import torch.nn.functional as F

scores = torch.tensor([[2.0, 1.0, 0.1],   # pizza, curry, salad
                       [0.5, 0.5, 3.0]])  # a second night
right = torch.tensor([0, 2])              # pizza, then salad
print(F.cross_entropy(scores[:1], right[:1]))  # tensor(0.4170)
print(F.cross_entropy(scores, right))     # tensor(0.2845)

## 6.8 Project: a network you wired by hand

Setting weights by hand shows exactly what training will do for you in Module 7, and why the hidden layer and its bend are needed.

[Read the lesson](https://aicourses.dev/research/neural-networks/project-network-by-hand)

**This is the project.** These cells are starting points from the lesson; the steps between them are yours to write. A cell that uses a name from one of your steps stops with a NameError until you have written that step.

### Steps

1. **Set up the four cases.** Open a new Colab notebook and start from this cell. With every weight at zero it runs, and gets two of the four wrong.

In [ ]:
import numpy as np

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])   # the four cases
y = np.array([0, 1, 1, 0])                       # XOR of each

W1 = np.zeros((2, 2))   # 2 hidden neurons × 2 inputs: your choice
b1 = np.zeros(2)        # one bias per hidden neuron
w2 = np.zeros(2)        # the output neuron’s two weights
b2 = 0.0                # and its bias

def forward(X):
    H = np.maximum(0, X @ W1.T + b1)   # hidden layer, then ReLU
    return H @ w2 + b2, H              # output and hidden values

out, H = forward(X)
print(out)                            # [0. 0. 0. 0.]
print(((out - y) ** 2).mean())        # mean squared error: 0.5
pred = (out > 0.5).astype(int)        # above 0.5 counts as “on”
print(pred, "all four right:", bool((pred == y).all()))
# [0 0 0 0] all four right: False

2. **Design hidden neuron 1.** Make it count how many switches are up: 0, 1, 1 and 2 for the four cases. Hint: give both inputs the same weight and no bias. Print `H` to check.
3. **Design hidden neuron 2.** Make it switch on only when both switches are up: 0, 0, 0 and 1. Hint: same weights as neuron 1, and a negative bias big enough that one switch alone cannot get past zero.
4. **Combine them in the output.** Neuron 1 says “at least one”, but it says 2 when both are up. Choose `w2` so neuron 2 cancels that: the outputs should come out 0, 1, 1, 0. Work each case on paper before you run it.
5. **Compute the loss of your weights.** The mean squared error line prints it. Perfect weights give exactly 0.0. Try a slightly wrong weight and see the loss rise above zero.
6. **Plot what each hidden neuron detects.** Add this as a new cell at the end:

In [ ]:
import matplotlib.pyplot as plt

g = np.linspace(-0.5, 1.5, 201)      # a fine grid around the square
gx, gy = np.meshgrid(g, g)
grid = np.column_stack([gx.ravel(), gy.ravel()])
_, Hg = forward(grid)                    # hidden values everywhere

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for i, ax in enumerate(axes):
    ax.contourf(gx, gy, Hg[:, i].reshape(gx.shape), levels=20)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm",
               edgecolors="black", s=120)
    ax.set_title(f"hidden neuron {i + 1}")
    ax.set_xlabel("switch 1")
    ax.set_ylabel("switch 2")
plt.savefig("xor_hidden.png")